# Data overview: homes and visits

Starting point for the third-places project. It shows where the data is and how to load it; the analysis is yours.

**Twitter tables** (PostgreSQL `twitter_cities_v2`, schema `revised`; every rule is documented in
`code/05_home_processing/home_pipeline/DECISIONS.md`):

| table | one row per | what it is |
|---|---|---|
| `home_final` | (city, user_id, res) | home H3 cell of a user, res 10 (~65 m) and res 9 (~175 m); `confidence` high / medium |
| `visit` | (city, user_id, local_date, cell_10) | **third-place candidates**: a user was in a res 10 cell on a day (home area and work cell removed) |
| `visit_user` | (city, user_id) | one row per visiting user: activity counts and the **`automated`** (bot) flag |

All tweets are 2013-2017, GPS-tagged, inside the city box, no retweets; times are local.

**External data** (repo root `external_data/`, each folder has a README):

| file | content |
|---|---|
| `population/pop_h3.csv` | residents per H3 cell, res 8/9/10, all three cities (2015) |
| `income/income_h3.csv` | household income per H3 cell, res 8/9/10, **London only** (2015/16) |
| `income/msoa_income_2016.csv` | the same income per MSOA (the level it is measured at) |

---
# 0. How to query the database from a notebook

Three steps, that is all you need:

1. **Connect.** The login details (host, user, password, ...) are in `code/03_database_setup/connection.json`.
   This file is not on GitHub; ask for it. We read it and build an `engine`: the object that knows how to talk to
   the database. You create it once per notebook.
2. **Write a query.** A query is just a Python string that contains SQL.
3. **Run it with pandas.** `pd.read_sql(query, engine)` sends the SQL to the database and returns the result as a
   normal pandas DataFrame.

Golden rule: the tables are big (`visit` has millions of rows), so **always add `limit`** when you just want to
look, and let the database do filtering (`where`) and counting (`count`, `group by`) instead of loading everything.

In [ ]:
# ── imports: pandas + sqlalchemy is all we need for the database ──────────────
import json
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# ── read the login details from the credentials file ──────────────────────────
cfg = json.load(open("../03_database_setup/connection.json"))

# ── build the connection (the "engine") ───────────────────────────────────────
url = URL.create(
    "postgresql+psycopg2",
    username=cfg["user"],
    password=cfg["password"],
    host=cfg["host"],
    port=cfg["port"],
    database=cfg["database"],
)
engine = create_engine(url)

**First query:** the first 5 rows of the home table. `revised.home_final` means table `home_final` in schema
`revised`.

In [ ]:
# ── write the query as a string ───────────────────────────────────────────────
query = """
select *
from revised.home_final
limit 5
"""

# ── run it: the result is a pandas DataFrame ──────────────────────────────────
homes_head = pd.read_sql(query, engine)
homes_head

**Structure:** column names and types of what we loaded, and the size of the full table (counted inside the
database, so nothing big is loaded).

In [ ]:
# ── column names and data types ───────────────────────────────────────────────
homes_head.info()

# ── number of rows in the full table ──────────────────────────────────────────
pd.read_sql("select count(*) as n_rows from revised.home_final", engine)

The same for the two visit tables. Only the table name changes.

In [ ]:
# ── first rows of the visit table ─────────────────────────────────────────────
pd.read_sql("select * from revised.visit limit 5", engine)

In [ ]:
# ── first rows of the per-user visit summary (with the bot flag) ──────────────
pd.read_sql("select * from revised.visit_user limit 5", engine)

**Filter in SQL:** `where` keeps only the rows you need, e.g. high-confidence London homes at res 9. Note that
text values in SQL use single quotes (`'london'`).

In [ ]:
# ── London homes at res 9, high confidence only ───────────────────────────────
pd.read_sql("""
select user_id, cell, confidence
from revised.home_final
where city = 'london' and res = 9 and confidence = 'high'
limit 5
""", engine)

---
# 1. Population

From here on we also make maps, so we need a few more packages. `hex_gdf` turns a table of H3 cell ids into
hexagon shapes; `hex_map` draws them on a light background map.

In [ ]:
# ── imports for maps and plots ────────────────────────────────────────────────
from pathlib import Path

import h3
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from shapely.geometry import Polygon

FS = 10
FS2 = FS - 2
FS4 = FS - 4
plt.rcParams.update({"font.size": FS})

# ── paths to the external data ────────────────────────────────────────────────
EXT = Path("../../external_data")
POP_CSV = EXT / "population" / "pop_h3.csv"
INCOME_CSV = EXT / "income" / "income_h3.csv"
MSOA_CSV = EXT / "income" / "msoa_income_2016.csv"

CITIES = ["amsterdam", "london", "portland"]
CITY_LABEL = {"amsterdam": "Amsterdam", "london": "London", "portland": "Portland"}

# ── helpers ───────────────────────────────────────────────────────────────────
# h3 v4 renamed h3_to_geo_boundary to cell_to_boundary; both return (lat, lng) pairs
H3_BOUNDARY = getattr(h3, "cell_to_boundary", None) or h3.h3_to_geo_boundary


def hex_gdf(df):
    # one hexagon polygon per row of the `cell` column, projected for web maps
    geom = [Polygon([(lng, lat) for lat, lng in H3_BOUNDARY(c)]) for c in df["cell"]]
    return gpd.GeoDataFrame(df, geometry=geom, crs="EPSG:4326").to_crs(epsg=3857)


def hex_map(ax, gdf, column, norm, cbar_label, label, cmap="Blues"):
    # hexagons coloured by `column`, background map, colorbar as tall as the map
    gdf.plot(column=column, cmap=cmap, norm=norm, linewidth=0, alpha=0.85, ax=ax)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, attribution_size=FS4)
    ax.set_axis_off()
    ax.text(0.0, 1.02, label, transform=ax.transAxes, fontsize=FS, fontweight="bold")
    cax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.05)
    cb = plt.gcf().colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), cax=cax)
    cb.set_label(cbar_label, fontsize=FS2)
    cb.ax.tick_params(labelsize=FS2)

Residents per H3 cell (GHS-POP 2015). It covers the whole city box, which is larger than the city itself (London
box 9.8M vs Greater London 8.6M). Res 10 values are noisy; res 9 or 8 is safer. The table below shows, per city
and resolution, the number of cells, total residents and the median cell.

In [ ]:
# ── load population (a plain csv, no database needed) ────────────────────────
pop = pd.read_csv(POP_CSV, dtype={"cell": str})

# ── cells, total residents and median per city and resolution ────────────────
pop.groupby(["city", "res"])["pop"].agg(["count", "sum", "median"]).round(0)

In [ ]:
# ── map: residents per res 9 cell, one panel per city (log colour scale) ──────
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, city, panel in zip(axes, CITIES, ["(A)", "(B)", "(C)"]):
    g = hex_gdf(pop[(pop["city"] == city) & (pop["res"] == 9)])
    hex_map(ax, g, "pop", LogNorm(1, g["pop"].max()), "residents per H3 res 9 cell",
            f"{panel} {CITY_LABEL[city]}", cmap="Greens")
plt.tight_layout()

---
# 2. Income (London)

Mean household income per MSOA (small areas of ~8,000 residents), 2015/16. We mostly use `income_net_bhc`: net
income adjusted for household size, before housing costs. The H3 cell values are blended from the MSOAs a cell
overlaps, so finer cells do not add detail. The estimates are uncertain: use income groups (quartiles), not exact
values. `greater_london` marks cells mostly inside Greater London.

In [ ]:
# ── load income per MSOA and per H3 cell ──────────────────────────────────────
msoa = pd.read_csv(MSOA_CSV)
inc = pd.read_csv(INCOME_CSV, dtype={"cell": str})

# ── summary of the four income measures across MSOAs ──────────────────────────
msoa[["income_total", "income_net", "income_net_bhc", "income_net_ahc"]].describe().round(0)

In [ ]:
# ── map: income per res 9 cell (colours clipped at the 2nd / 98th percentile) ─
g = hex_gdf(inc[inc["res"] == 9])
norm = Normalize(g["income_net_bhc"].quantile(0.02), g["income_net_bhc"].quantile(0.98))
fig, ax = plt.subplots(figsize=(9, 6.5))
hex_map(ax, g, "income_net_bhc", norm,
        "net equivalised household income before housing costs (GBP, 2015/16)", "(A) London, res 9",
        cmap="viridis")
plt.tight_layout()

## 2.1 Income groups

One way to define income groups: quartiles of the res 9 cells in Greater London, weighted by population so that
each quartile holds a quarter of the residents (1 = poorest, 4 = richest). This is an example; the exact choice is
part of your design.

In [ ]:
# ── Greater London res 9 cells, sorted from poor to rich ──────────────────────
gl9 = inc[(inc["res"] == 9) & inc["greater_london"]].sort_values("income_net_bhc")

# ── cut points where 25 / 50 / 75% of the residents are reached ───────────────
cum = gl9["pop"].cumsum() / gl9["pop"].sum()
cuts = [gl9.loc[cum >= q, "income_net_bhc"].iloc[0] for q in [0.25, 0.5, 0.75]]

# ── quartile 1-4 for every cell ───────────────────────────────────────────────
gl9["income_q"] = 1 + np.searchsorted(cuts, gl9["income_net_bhc"], side="right")
print("cut points (GBP):", [int(c) for c in cuts])
gl9.groupby("income_q").agg(cells=("cell", "count"), pop=("pop", "sum"))

---
# 3. Homes and visits

**Always drop automated accounts.** A few bot-like feeds (job ads, traffic) post from many places every day: in
Amsterdam and Portland 10-16 accounts hold a third to two thirds of all raw visits. The pattern is always the
same: join `visit_user` on (city, user_id) and keep `not u.automated`.

In [ ]:
# ── home users per city (res 9; every user has one row per resolution) ───────
pd.read_sql("""
select city
     , count(*)                                      as n_home_users
     , count(*) filter (where confidence = 'high')   as n_high_confidence
from revised.home_final
where res = 9
group by city
order by city
""", engine)

In [ ]:
# ── visits per city, automated accounts removed ──────────────────────────────
pd.read_sql("""
select v.city
     , count(*)                    as n_visits
     , count(distinct v.user_id)   as n_users
     , count(distinct v.cell_10)   as n_cells
from revised.visit v
join revised.visit_user u
  on u.city = v.city and u.user_id = v.user_id
where not u.automated
group by v.city
order by v.city
""", engine)

## 3.1 Example join: home income of London users

Homes join to income on the H3 cell (res 9 in both). How do home users spread over the income quartiles, compared
with the residents? Twitter users are not a random sample (~1-2% of residents, younger, more central), so expect a
skew.

In [ ]:
# ── London homes at res 9 (cell as text so it matches the csv) ────────────────
homes = pd.read_sql("""
select user_id, cell::text as cell
from revised.home_final
where city = 'london' and res = 9
""", engine)

# ── attach the income quartile of the home cell (Greater London only) ────────
homes = homes.merge(gl9[["cell", "income_q"]], on="cell", how="inner")

# ── share of home users vs share of residents per quartile ───────────────────
pd.DataFrame({
    "share_homes": homes["income_q"].value_counts(normalize=True).sort_index(),
    "share_residents": gl9.groupby("income_q")["pop"].sum() / gl9["pop"].sum(),
}).round(3)

## 3.2 Example: most visited places in London

Visits are stored at res 10; `h3_cell_to_parent(cell, 9)` moves them up to the larger res 9 cell. The visitor's
home (and so income) can be added from `home_final` via `user_id`. Matching cells to amenities (POIs) is the next
step.

In [ ]:
# ── top 10 res 9 cells by number of distinct visitors ─────────────────────────
pd.read_sql("""
select public.h3_cell_to_parent(v.cell_10, 9)::text   as cell_9
     , count(distinct v.user_id)                      as n_visitors
     , count(*)                                       as n_visits
from revised.visit v
join revised.visit_user u
  on u.city = v.city and u.user_id = v.user_id
where v.city = 'london' and not u.automated
group by 1
order by n_visitors desc
limit 10
""", engine)

---
# 4. Caveats to keep in mind

- **Sample:** homes exist for ~150k users (~1-2% of residents); heavy users and central areas are over-represented.
- **Automated accounts:** always filter `not visit_user.automated`.
- **Activity:** users who tweet more visit more places. Normalise per-user counts by active days
  (`visit_user.cells_per_10_days`) and keep users with at least ~10 active days; otherwise income effects can just
  be tweeting effects.
- **A visit is a tweet location**, not a confirmed stay: some visits are tweets in transit (rail lines, roads).
  GPS error is 10-50 m, so a res 10 cell (~65 m) is the finest sensible unit.
- **Home area removed:** the home cell and its neighbours (~200 m) are not in `visit`.
- **Income** is a model estimate per MSOA; use groups, not exact values.